In [9]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import TensorDataset, DataLoader

# Kontrollera om en GPU (CUDA) finns
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Tränar på: {device}")

class LSTMAutoencoder(nn.Module):
    def __init__(self, seq_len, n_features, hidden_dim=64):
        super(LSTMAutoencoder, self).__init__()
        self.seq_len = seq_len
        self.n_features = n_features
        self.hidden_dim = hidden_dim

        # --- ENCODER ---
        self.encoder = nn.LSTM(
            input_size=n_features, 
            hidden_size=hidden_dim, 
            num_layers=1, 
            batch_first=True
        )
        
        # --- DECODER ---
        self.decoder = nn.LSTM(
            input_size=hidden_dim, 
            hidden_size=hidden_dim, 
            num_layers=1, 
            batch_first=True
        )
        
        # --- OUTPUT LAYER ---
        self.output_layer = nn.Linear(hidden_dim, n_features)

    def forward(self, x):
        _, (hidden, _) = self.encoder(x)
        encoded = hidden[-1].unsqueeze(1).repeat(1, self.seq_len, 1)
        decoded, _ = self.decoder(encoded)
        reconstructed = self.output_layer(decoded)
        return reconstructed


# --- 1. DATAFÖRBEREDELSE ---
testdata = "/mnt/c/Users/linne/Skola/Deep-Learning-DV2646/Uppgift 2/Train_data.csv"
df = pd.read_csv(testdata)

# Filtrera ut endast numerisk data för att koden ska fungera direkt.
# OBS: I ett riktigt projekt bör du One-Hot-enkoda kategorier (protocol_type, service, flag)
# och du måste filtrera så du BARA tränar på rader där trafiken är "normal".
df_numeric = df.select_dtypes(include=[np.number]).dropna()

# Skala datan (Kritiskt för LSTM)
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(df_numeric)

# --- 2. SKAPA 3D-SEKVENSER (Sliding Windows) ---
def create_sequences(data, seq_length):
    xs = []
    # Skapar fönster över datan
    for i in range(len(data) - seq_length):
        xs.append(data[i:(i + seq_length)])
    return np.array(xs)

seq_len = 10
X_seq = create_sequences(data_scaled, seq_len)

# --- 3. SKAPA DATALOADER ---
# Konvertera Numpy-array till PyTorch Tensors
X_tensor = torch.tensor(X_seq, dtype=torch.float32)

# Skapa ett dataset och en dataloader (batch_size=64 är vanligt att börja med)
dataset = TensorDataset(X_tensor)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

# Uppdatera n_features baserat på den faktiska formen på vår data
n_features = X_seq.shape[2] 
print(f"Data formaterad. Input shape är: (Batch, {seq_len}, {n_features})")


# --- 4. INITIERA MODELL OCH TRÄNING ---
model = LSTMAutoencoder(seq_len, n_features).to(device)
criterion = nn.L1Loss() # MAE
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 50

for epoch in range(epochs):
    model.train() 
    train_loss = 0
    
    # TensorDataset returnerar en tuple, därför (batch_x,)
    for (batch_x,) in train_loader:
        batch_x = batch_x.to(device)
        
        optimizer.zero_grad()
        
        reconstructed = model(batch_x)
        loss = criterion(reconstructed, batch_x)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    avg_loss = train_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.6f}')

Tränar på: cuda
Data formaterad. Input shape är: (Batch, 10, 38)
Epoch 1/50, Loss: 0.112133
Epoch 2/50, Loss: 0.106108
Epoch 3/50, Loss: 0.105353
Epoch 4/50, Loss: 0.105047
Epoch 5/50, Loss: 0.104855
Epoch 6/50, Loss: 0.104323
Epoch 7/50, Loss: 0.103699
Epoch 8/50, Loss: 0.102100
Epoch 9/50, Loss: 0.099093
Epoch 10/50, Loss: 0.091436
Epoch 11/50, Loss: 0.082932
Epoch 12/50, Loss: 0.077479
Epoch 13/50, Loss: 0.072249
Epoch 14/50, Loss: 0.067905
Epoch 15/50, Loss: 0.064623
Epoch 16/50, Loss: 0.062180
Epoch 17/50, Loss: 0.060157
Epoch 18/50, Loss: 0.057692
Epoch 19/50, Loss: 0.054659
Epoch 20/50, Loss: 0.052172
Epoch 21/50, Loss: 0.049238
Epoch 22/50, Loss: 0.045230
Epoch 23/50, Loss: 0.041446
Epoch 24/50, Loss: 0.039655
Epoch 25/50, Loss: 0.038645
Epoch 26/50, Loss: 0.037863
Epoch 27/50, Loss: 0.036976
Epoch 28/50, Loss: 0.036398
Epoch 29/50, Loss: 0.035767
Epoch 30/50, Loss: 0.035289
Epoch 31/50, Loss: 0.034847
Epoch 32/50, Loss: 0.034551
Epoch 33/50, Loss: 0.034287
Epoch 34/50, Loss: 0